# LC3D — photo → forme organique (gratuit, GPU Kaggle)

Ce carnet roule sur **Kaggle** (compte gratuit, 30 h de GPU par semaine). Il transforme une photo en forme 3D avec **Hunyuan3D-2** (Tencent, libre), forme seule (pas de couleurs : on imprime).

1. Kaggle → *Create* → *New Notebook* → *File* → *Import notebook* → ce fichier.
2. À droite : *Settings* → *Accelerator* → **GPU T4 x2** ; *Internet* → **On**.
3. *Add data* → *Upload* : ta photo (fond simple, objet entier, bien éclairé).
4. *Run All*. Le fichier `forme.glb` apparaît dans *Output* : télécharge-le et envoie-le à Claude (LC3D le répare, le met à la taille et lui ajoute ses fonctions : aimants, cachette, LED, socle…).

_Non testé dans l'espace de Claude (pas de GPU ni d'accès à Hugging Face) : premier usage à confirmer._

In [ ]:
!pip -q install diffusers einops omegaconf transformers accelerate pymeshlab pygltflib xatlas trimesh opencv-python ninja pybind11
!pip -q install git+https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git

In [ ]:
import glob, os
PHOTO = sorted(glob.glob('/kaggle/input/**/*.[jp][pn]g', recursive=True) + glob.glob('/kaggle/input/**/*.jpeg', recursive=True))[0]
print('Photo :', PHOTO)

In [ ]:
from PIL import Image
from hy3dgen.rembg import BackgroundRemover
img = Image.open(PHOTO).convert('RGB')
img = BackgroundRemover()(img)   # enlève le fond
img.save('/kaggle/working/sans_fond.png'); img

In [ ]:
import torch
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
# version « mini » en fp16 : forme seule ≈ 6 Go de mémoire vidéo (T4 = 16 Go)
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2mini', subfolder='hunyuan3d-dit-v2-mini', variant='fp16')
mesh = pipe(image=img, num_inference_steps=50, octree_resolution=380, num_chunks=20000, generator=torch.manual_seed(12345), output_type='trimesh')[0]

In [ ]:
from hy3dgen.shapegen import FaceReducer, FloaterRemover, DegenerateFaceRemover
mesh = FloaterRemover()(mesh); mesh = DegenerateFaceRemover()(mesh); mesh = FaceReducer()(mesh, max_facenum=200000)
mesh.export('/kaggle/working/forme.glb'); mesh.export('/kaggle/working/forme.stl')
print('Fini : forme.glb et forme.stl dans Output')